In [1]:
!pip install gradio transformers torch tf-keras datasets tqdm evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00


In [2]:
import gradio as gr
from transformers import pipeline
from datasets import load_dataset

print("Loading SQuAD dataset...")
# Load SQuAD v1.1 dataset
squad_dataset = load_dataset("squad", split="train")
print(f"Loaded {len(squad_dataset)} examples from SQuAD dataset")

# Extract first 10 examples (context, question, and answer)
squad_examples = []
for i in range(10):
    example = squad_dataset[i]
    # Get the answer text (first answer if multiple exist)
    answer_text = example['answers']['text'][0] if example['answers']['text'] else ""
    squad_examples.append([
        example['context'],
        example['question'],
        answer_text  # Ground truth answer from dataset
    ])
print(f"Extracted first {len(squad_examples)} examples with answers")

print("Loading models...")

# --- Model 1: DistilBERT (Fast, SQuAD v1.1) ---
# Good for speed, trained on answerable questions only.
model_1_name = "distilbert-base-cased-distilled-squad"
qa_pipeline_1 = pipeline("question-answering", model=model_1_name)

# --- Model 2: Distilled RoBERTa (Accurate, SQuAD v2.0) ---
# Good for accuracy, can handle "unanswerable" questions (returns empty string).
model_2_name = "deepset/roberta-base-squad2-distilled"
qa_pipeline_2 = pipeline("question-answering", model=model_2_name)

Loading SQuAD dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0


In [3]:
# --- Model Evaluation on SQuAD Dataset ---

from evaluate import evaluator
from datasets import load_dataset

print("\n" + "="*60)
print("EVALUATING MODELS ON SQuAD DATASET")
print("="*60)

# Load entire validation set for evaluation
eval_data = load_dataset("squad", split="validation")
print(f"Loaded {len(eval_data)} examples from SQuAD validation set")

# Initialize the question-answering evaluator
task_evaluator = evaluator("question-answering")

# Evaluate Model 1 (DistilBERT - SQuAD v1.1)
print(f"\nEvaluating {model_1_name}...")
results_1 = task_evaluator.compute(
    model_or_pipeline=qa_pipeline_1,
    data=eval_data,
    metric="squad",
)

# Evaluate Model 2 (RoBERTa - SQuAD v2.0)
# Note: Model 2 is trained on SQuAD v2, but we're evaluating on SQuAD v1.1
# For SQuAD v2 evaluation, use squad_v2_format=True
print(f"\nEvaluating {model_2_name}...")
results_2 = task_evaluator.compute(
    model_or_pipeline=qa_pipeline_2,
    data=eval_data,
    metric="squad",
)



EVALUATING MODELS ON SQuAD DATASET
Loaded 10570 examples from SQuAD validation set

Evaluating distilbert-base-cased-distilled-squad...


Filter:   0%|          | 0/10570 [00:00<?, ? examples/s]


Evaluating deepset/roberta-base-squad2-distilled...


Filter: 100%|##########| 10570/10570 [00:00<?, ? examples/s]

In [4]:
# Display results
print("\n" + "="*60)
print("EVALUATION RESULTS (L4 GPU)")
print("="*60)
print(f"\nModel 1: {model_1_name}")
print(f"  F1 Score: {results_1['f1']:.4f}")
print(f"  Exact Match: {results_1['exact_match']:.4f}")
print(f"  Total Time (seconds): {results_1['total_time_in_seconds']:.2f}")
print(f"  Samples Per Second: {results_1['samples_per_second']:.2f}")
print(f"  Latency Per Sample (seconds): {results_1['latency_in_seconds']:.4f}")


print(f"\nModel 2: {model_2_name}")
print(f"  F1 Score: {results_2['f1']:.4f}")
print(f"  Exact Match: {results_2['exact_match']:.4f}")
print(f"  Total Time (seconds): {results_2['total_time_in_seconds']:.2f}")
print(f"  Samples Per Second: {results_2['samples_per_second']:.2f}")
print(f"  Latency Per Sample (seconds): {results_2['latency_in_seconds']:.4f}")



EVALUATION RESULTS (L4 GPU)

Model 1: distilbert-base-cased-distilled-squad
  F1 Score: 86.8528
  Exact Match: 79.5837
  Total Time (seconds): 71.26
  Samples Per Second: 148.33
  Latency Per Sample (seconds): 0.0067

Model 2: deepset/roberta-base-squad2-distilled
  F1 Score: 92.3934
  Exact Match: 85.9697
  Total Time (seconds): 114.11
  Samples Per Second: 92.63
  Latency Per Sample (seconds): 0.0108


In [5]:
def compare(metric, label):
    diff = abs(results_1[metric] - results_2[metric])

    # Metric comparison
    if results_1[metric] > results_2[metric]:
        print(f"🏆 Model 1 ({model_1_name}) has higher {label} by {diff:.4f}")
    elif results_2[metric] > results_1[metric]:
        print(f"🏆 Model 2 ({model_2_name}) has higher {label} by {diff:.4f}")
    else:
        print(f"🤝 Models have similar {label} scores")

def compare_latency():
    lat1 = results_1['latency_in_seconds']
    lat2 = results_2['latency_in_seconds']

    if lat1 < lat2:  # lower latency = faster
        pct = ((lat2 - lat1) / lat1) * 100
        print(f"⚡ Model 1 ({model_1_name}) is faster by {pct:.2f}%")
    elif lat2 < lat1:
        pct = ((lat1 - lat2) / lat2) * 100
        print(f"⚡ Model 2 ({model_2_name}) is faster by {pct:.2f}%")
    else:
        print("🤝 Both models have similar processing speeds")


print("COMPARISON")
print("=" * 60)

compare('f1', 'F1 Score')
compare('exact_match', 'Exact Match')
compare_latency()

print("=" * 60 + "\n")

COMPARISON
🏆 Model 2 (deepset/roberta-base-squad2-distilled) has higher F1 Score by 5.5406
🏆 Model 2 (deepset/roberta-base-squad2-distilled) has higher Exact Match by 6.3860
⚡ Model 1 (distilbert-base-cased-distilled-squad) is faster by 60.13%



In [8]:
# Gradio App

def compare_models(context, question, ground_truth_answer_unused=None):
    # Error handling for empty inputs
    if not context or not question:
        return "N/A", 0.0, "N/A", 0.0

    # Run Model 1
    res1 = qa_pipeline_1(question=question, context=context)

    # Run Model 2
    res2 = qa_pipeline_2(question=question, context=context)

    # Return: Ans1, Conf1, Ans2, Conf2
    return res1['answer'], res1['score'], res2['answer'], res2['score']

# --- Gradio UI Layout ---
with gr.Blocks(title="QA Model Compare") as demo:
    gr.Markdown("# QA Model Compare: DistilBERT vs. Distlled RoBERTa")
    gr.Markdown("Compare a lightweight model (DistilBERT) against a more robust model (Distilled RoBERTa) on the **SQuAD** dataset.")

    with gr.Row():
        with gr.Column(scale=1):
            context_input = gr.Textbox(lines=8, label="Context Paragraph", placeholder="Paste context here...")
            question_input = gr.Textbox(lines=2, label="Question")
            ground_truth_answer = gr.Textbox(lines=2, label="Ground Truth Answer (from SQuAD)", interactive=False)
            submit_btn = gr.Button("Compare Models", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown(f"### Model 1: {model_1_name}")
            out_ans_1 = gr.Textbox(label="Answer")
            out_conf_1 = gr.Number(label="Confidence")

            gr.Markdown("---")

            gr.Markdown(f"### Model 2: {model_2_name}")
            out_ans_2 = gr.Textbox(label="Answer")
            out_conf_2 = gr.Number(label="Confidence")

    # Link inputs/outputs
    submit_btn.click(
        fn=compare_models,
        inputs=[context_input, question_input],
        outputs=[out_ans_1, out_conf_1, out_ans_2, out_conf_2]
    )

    # Add Examples
    gr.Examples(
        examples=squad_examples,
        inputs=[context_input, question_input, ground_truth_answer],
        outputs=[out_ans_1, out_conf_1, out_ans_2, out_conf_2],
        fn=compare_models,
        cache_examples=True,
    )

if __name__ == "__main__":
    demo.launch()

Caching examples at: '/content/.gradio/cached_examples/41'
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6072644dc7b3902125.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
